<a href="https://colab.research.google.com/github/cleberte88/comissoes_vendedores/blob/main/comissoesInternos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTAÇÃO DAS BIBLIOTECAS NECESSÁRIAS PRO PROJETO
from pyspark.sql import SparkSession as ss
from pyspark.sql import functions as f
from pyspark.sql.types import DoubleType
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.7 MB/s eta 0:00:00


In [ ]:
#Criando o app que valida a comissão
spark = ss.builder.appName("comissoesInternos").getOrCreate()

In [ ]:
#Transformando os arquivos CSV em algo legível no spark
url_comiss = "/content/drive/MyDrive/comissao_internos/comissao_item_.csv"
url_dev = "/content/drive/MyDrive/comissao_internos/Comissao_Item_Dev_.csv"
url_ped = "/content/drive/MyDrive/comissao_internos/analise_ped_.csv"
url_fat = "/content/drive/MyDrive/comissao_internos/analise_fat_.csv"
url_dist = "/content/drive/MyDrive/comissao_internos/DISTRIBUIDORES (3).csv"

comiss_item = spark.read.csv(
    url_comiss,
    sep = ",",
    header = True,
    inferSchema = True
)
comiss_dev = spark.read.csv(
    url_dev,
    sep = ",",
    header = True,
    inferSchema = True
)
analise_fat = spark.read.csv(
    url_fat,
    sep = ",",
    header = True,
    inferSchema = True
)
analise_ped = spark.read.csv(
    url_ped,
    sep = ",",
    header = True,
    inferSchema = True
)
distribuidor = spark.read.csv(
    url_dist,
    sep = ",",
    header = True,
    inferSchema = True
)


In [ ]:
distribuidor.printSchema()

root
 |-- DISTRIBUIDORES: string (nullable = true)
 |--  SAP: string (nullable = true)
 |--  USO PRINCIPAL: string (nullable = true)
 |--  PRAZO ATUAL: string (nullable = true)
 |--  SISTEMA: string (nullable = true)
 |--  Local: string (nullable = true)



In [ ]:
comiss_dev.printSchema()

root
 |-- cardcode: string (nullable = true)
 |-- cardname: string (nullable = true)
 |-- NumDev: integer (nullable = true)
 |-- NFDev: integer (nullable = true)
 |-- Data Dev: string (nullable = true)
 |-- Número Interno Nota Fiscal de Saída: integer (nullable = true)
 |-- Nota Fiscal: integer (nullable = true)
 |-- Status Devolução: string (nullable = true)
 |-- ItemCode: string (nullable = true)
 |-- Dscription: string (nullable = true)
 |-- UomCode: string (nullable = true)
 |-- Quantidade Dev: double (nullable = true)
 |-- Valor: double (nullable = true)
 |-- Comissão no PV: integer (nullable = true)
 |-- Tabela Promocional: double (nullable = true)
 |-- R$ Tabela Comissão: double (nullable = true)
 |-- % Tabela Comissão: double (nullable = true)
 |-- Comissão Padrão 3%: double (nullable = true)
 |-- Valor Comissão Devolução: double (nullable = true)
 |-- Representante: string (nullable = true)
 |-- LineNum: integer (nullable = true)



In [ ]:
#Testando o DataFrame para analisar se precisa de algum ajuste
comiss_item.show(2, False)

+----------+------------------------+---------------+----------+---------+-------------+-------------+-------+--------------------------------+----+----+-------------+--------------+------------------+---------+------------------+-----------------+------------------+------------------+------------+
|CodCliente|cliente                 |Pedido de Venda|NotaFiscal|documento|DataDocumento|Representante|CodItem|item                            |UNNF|Qtd |ValorContabil|Comissão no PV|Tabela Promocional|U_CodItem|R$ Tabela Comissão|% Tabela Comissão|Comissão Padrão 3%|Total do documento|Tipo do Item|
+----------+------------------------+---------------+----------+---------+-------------+-------------+-------+--------------------------------+----+----+-------------+--------------+------------------+---------+------------------+-----------------+------------------+------------------+------------+
|CL008563  |FABRICIA ROWEDER ANTONIO|NULL           |2172      |2042678  |02/10/2026   |MAR E RIO   

In [ ]:
#Aqui estou vendo os tipos de dados que tenho, para conseguir manipular da forma correta
comiss_item.printSchema()
#comiss_dev.printSchema()
#analise_fat.printSchema()
#analise_ped.printSchema()

root
 |-- CodCliente: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- Pedido de Venda: integer (nullable = true)
 |-- NotaFiscal: integer (nullable = true)
 |-- documento: integer (nullable = true)
 |-- DataDocumento: string (nullable = true)
 |-- Representante: string (nullable = true)
 |-- CodItem: string (nullable = true)
 |-- item: string (nullable = true)
 |-- UNNF: string (nullable = true)
 |-- Qtd: double (nullable = true)
 |-- ValorContabil: double (nullable = true)
 |-- Comissão no PV: integer (nullable = true)
 |-- Tabela Promocional: double (nullable = true)
 |-- U_CodItem: string (nullable = true)
 |-- R$ Tabela Comissão: double (nullable = true)
 |-- % Tabela Comissão: double (nullable = true)
 |-- Comissão Padrão 3%: double (nullable = true)
 |-- Total do documento: double (nullable = true)
 |-- Tipo do Item: string (nullable = true)



In [ ]:
#Aqui defini os DF para variáveis que irei trabalhar no código (redundante)
df_com = comiss_item
df_dev = comiss_dev
df_fat = analise_fat
df_ped = analise_ped

In [ ]:
#Renomeando as colunas para conseguir trabalhar no modelo SQL Spark
df_com = df_com.withColumnRenamed("Comissão no PV", "comissao_no_pv")\
.withColumnRenamed("Pedido de Venda", "pedido_venda")\
.withColumnRenamed("Tabela Promocional", "tabela_promocional")\
.withColumnRenamed("R$ Tabela Comissão", "RS_tabela_comissao")\
.withColumnRenamed("% Tabela Comissão", "percent_comissao")\
.withColumnRenamed("Comissão Padrão 3%", "comissao_padrao_3perc")\
.withColumnRenamed("Total do documento", "total_documento")\
.withColumnRenamed("Tipo do Item", "tipo_item")

In [ ]:
# Convertendo o formato da coluna de data que estava como string, assim como o de
# valor de comissão que estava também como string
df_com.withColumn("DataDocumento", f.to_date(df_com["DataDocumento"], "MM/dd/yyyy"))\
.withColumn("comissao_no_pv", f.col("comissao_no_pv").cast(DoubleType())).show(2, False)

+----------+------------------------+------------+----------+---------+-------------+-------------+-------+--------------------------------+----+----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+
|CodCliente|cliente                 |pedido_venda|NotaFiscal|documento|DataDocumento|Representante|CodItem|item                            |UNNF|Qtd |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|
+----------+------------------------+------------+----------+---------+-------------+-------------+-------+--------------------------------+----+----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+
|CL008563  |FABRICIA ROWEDER ANTONIO|NULL        |2172      |2042678  |2026-02-10   |MAR E RIO    |6      |FILE MERLUZA A

In [ ]:
#atribuindo o ajuste ao DataFrame para ficar guardado a atualização dos dados
df_com = df_com.withColumn("DataDocumento", f.to_date(df_com["DataDocumento"], "MM/dd/yyyy"))\
.withColumn("comissao_no_pv", f.col("comissao_no_pv").cast(DoubleType()))

In [ ]:
df_dev.show(1, False)

+--------+-----------------------------+------+------+----------+-----------------------------------+-----------+----------------+--------+------------------------------------+-------+--------------+-------+--------------+------------------+------------------+-----------------+------------------+------------------------+-------------+-------+
|cardcode|cardname                     |NumDev|NFDev |Data Dev  |Número Interno Nota Fiscal de Saída|Nota Fiscal|Status Devolução|ItemCode|Dscription                          |UomCode|Quantidade Dev|Valor  |Comissão no PV|Tabela Promocional|R$ Tabela Comissão|% Tabela Comissão|Comissão Padrão 3%|Valor Comissão Devolução|Representante|LineNum|
+--------+-----------------------------+------+------+----------+-----------------------------------+-----------+----------------+--------+------------------------------------+-------+--------------+-------+--------------+------------------+------------------+-----------------+------------------+-------------

In [ ]:
# Renomeando as colunas para trabalhar no modelo SQL Spark
df_dev.withColumnRenamed("Data Dev", "data_dev")\
.withColumnRenamed("Número Interno Nota Fiscal de Saída", "nf_saida")\
.withColumnRenamed("Nota Fiscal", "NF")\
.withColumnRenamed("Status Devolução", "status_dev")\
.withColumnRenamed("Quantidade Dev", "qtd_dev")\
.withColumnRenamed("Comissão no PV", "comissao_no_pv")\
.withColumnRenamed("Tabela Promocional", "tabela_promocional")\
.withColumnRenamed("R$ Tabela Comissão", "RS_tabela_comissao")\
.withColumnRenamed("% Tabela Comissão", "perc_tabela_comissao")\
.withColumnRenamed("Comissão Padrão 3%", "comissao_padrao_3perc")\
.withColumnRenamed("Valor Comissão Devolução", "valor_comissao_dev").show(1, False)

+--------+-----------------------------+------+------+----------+--------+------+---------------+--------+------------------------------------+-------+-------+-------+--------------+------------------+------------------+--------------------+---------------------+------------------+-------------+-------+
|cardcode|cardname                     |NumDev|NFDev |data_dev  |nf_saida|NF    |status_dev     |ItemCode|Dscription                          |UomCode|qtd_dev|Valor  |comissao_no_pv|tabela_promocional|RS_tabela_comissao|perc_tabela_comissao|comissao_padrao_3perc|valor_comissao_dev|Representante|LineNum|
+--------+-----------------------------+------+------+----------+--------+------+---------------+--------+------------------------------------+-------+-------+-------+--------------+------------------+------------------+--------------------+---------------------+------------------+-------------+-------+
|CL000013|MARIA CRISTINA HISAE SAKAMOTO|40300 |105990|02/25/2026|2059039 |105579|Devo

In [ ]:
# Atribuindo as mudanças depois de analisar ao DataFrame
df_dev = df_dev.withColumnRenamed("Data Dev", "data_dev")\
.withColumnRenamed("Número Interno Nota Fiscal de Saída", "nf_saida")\
.withColumnRenamed("Nota Fiscal", "NF")\
.withColumnRenamed("Status Devolução", "status_dev")\
.withColumnRenamed("Quantidade Dev", "qtd_dev")\
.withColumnRenamed("Comissão no PV", "comissao_no_pv")\
.withColumnRenamed("Tabela Promocional", "tabela_promocional")\
.withColumnRenamed("R$ Tabela Comissão", "RS_tabela_comissao")\
.withColumnRenamed("% Tabela Comissão", "perc_tabela_comissao")\
.withColumnRenamed("Comissão Padrão 3%", "comissao_padrao_3perc")\
.withColumnRenamed("Valor Comissão Devolução", "valor_comissao_dev")

In [ ]:
# Analisando os tipos de dados para fazer as devidas limpezas
# Garantindo que as colunas a serem calculadas não dêem problemas
df_dev.printSchema()

root
 |-- cardcode: string (nullable = true)
 |-- cardname: string (nullable = true)
 |-- NumDev: integer (nullable = true)
 |-- NFDev: integer (nullable = true)
 |-- data_dev: string (nullable = true)
 |-- nf_saida: integer (nullable = true)
 |-- NF: integer (nullable = true)
 |-- status_dev: string (nullable = true)
 |-- ItemCode: string (nullable = true)
 |-- Dscription: string (nullable = true)
 |-- UomCode: string (nullable = true)
 |-- qtd_dev: double (nullable = true)
 |-- Valor: double (nullable = true)
 |-- comissao_no_pv: integer (nullable = true)
 |-- tabela_promocional: double (nullable = true)
 |-- RS_tabela_comissao: double (nullable = true)
 |-- perc_tabela_comissao: double (nullable = true)
 |-- comissao_padrao_3perc: double (nullable = true)
 |-- valor_comissao_dev: double (nullable = true)
 |-- Representante: string (nullable = true)
 |-- LineNum: integer (nullable = true)



In [ ]:
# Transformando os dados em datas e valores float (double)
df_dev = df_dev.withColumn("data_dev", f.to_date(df_dev["data_dev"], "MM/dd/yyyy"))\
.withColumn("comissao_no_pv", f.col("comissao_no_pv").cast(DoubleType()))

In [ ]:
df_fat.show(2, False)

+------------+----------+----+---------+----------+----------------+---------------------+---+------+---------+-----------+-------------+-------------+-------+------------------------+----+----+----------+-----------+----------------+-----+----------------+------------+---------------+-----+-------------------+
|Tipo        |NotaFiscal|Rota|documento|CodCliente|cliente         |cidade               |UF |Filial|Quadrante|GrupoItem  |DataDocumento|Representante|CodItem|Item                    |UNNF|Qtd |FatorVenda|FatorCompra|Quantidade Total|Valor|Custo do Item NF|Peso Líquido|Peso Bruto item|Custo|Utilizacao         |
+------------+----------+----+---------+----------+----------------+---------------------+---+------+---------+-----------+-------------+-------------+-------+------------------------+----+----+----------+-----------+----------------+-----+----------------+------------+---------------+-----+-------------------+
|Venda Varejo|2050      |NULL|2031653  |CL008027  |CONSUMIDOR

In [ ]:
# Atribuindo mudanças diretas no DataFrame
df_fat = df_fat.withColumnRenamed("Quantidade Total", "qtd_total")\
.withColumnRenamed("Custo do Item NF", "custo_do_item_nf")\
.withColumnRenamed("Peso Líquido", "peso_liquido")\
.withColumnRenamed("Peso Bruto item", "peso_bruto_item")

In [ ]:
df_fat.printSchema()

root
 |-- Tipo: string (nullable = true)
 |-- NotaFiscal: integer (nullable = true)
 |-- Rota: string (nullable = true)
 |-- documento: integer (nullable = true)
 |-- CodCliente: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- Filial: string (nullable = true)
 |-- Quadrante: string (nullable = true)
 |-- GrupoItem: string (nullable = true)
 |-- DataDocumento: string (nullable = true)
 |-- Representante: string (nullable = true)
 |-- CodItem: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- UNNF: string (nullable = true)
 |-- Qtd: double (nullable = true)
 |-- FatorVenda: integer (nullable = true)
 |-- FatorCompra: integer (nullable = true)
 |-- qtd_total: double (nullable = true)
 |-- Valor: double (nullable = true)
 |-- custo_do_item_nf: double (nullable = true)
 |-- peso_liquido: double (nullable = true)
 |-- peso_bruto_item: double (nullable = true)
 |-- Custo: double (nulla

In [ ]:
# Atualizando a formatação da data na coluna que estava como string
df_fat.withColumn("DataDocumento", f.to_date(df_fat["DataDocumento"], "MM/dd/yyyy")).show(1, False)

+------------+----------+----+---------+----------+----------------+---------------------+---+------+---------+-----------+-------------+-------------+-------+------------------------+----+----+----------+-----------+---------+-----+----------------+------------+---------------+-----+-------------------+
|Tipo        |NotaFiscal|Rota|documento|CodCliente|cliente         |cidade               |UF |Filial|Quadrante|GrupoItem  |DataDocumento|Representante|CodItem|Item                    |UNNF|Qtd |FatorVenda|FatorCompra|qtd_total|Valor|custo_do_item_nf|peso_liquido|peso_bruto_item|Custo|Utilizacao         |
+------------+----------+----+---------+----------+----------------+---------------------+---+------+---------+-----------+-------------+-------------+-------+------------------------+----+----+----------+-----------+---------+-----+----------------+------------+---------------+-----+-------------------+
|Venda Varejo|2050      |NULL|2031653  |CL008027  |CONSUMIDOR FINAL|São José do Ri

In [ ]:
# Atribuindo ao DataFrame as alterações feitas
df_fat = df_fat.withColumn("DataDocumento", f.to_date(df_fat["DataDocumento"], "MM/dd/yyyy"))

In [ ]:
df_ped.show(1, False)

+----------+-----------+---------+-------+----------+-------------------------------+--------+---+----------------------------------------------+---------+-------------------+-------------+-------------+--------+-------------------------------------------------+------+-----+----------+-----------+------------+-------------+------------+----------+
|NotaFiscal|Rota       |Documento|Pedido |CodCliente|Cliente                        |Cidade  |UF |Filial                                        |Quadrante|GrupoItem          |DataDocumento|Representante|CodItem |Item                                             |UNNF  |Qtd  |FatorVenda|FatorCompra|Valor Tabela|% de desconto|Peso Líquido|Peso Bruto|
+----------+-----------+---------+-------+----------+-------------------------------+--------+---+----------------------------------------------+---------+-------------------+-------------+-------------+--------+-------------------------------------------------+------+-----+----------+-----------+--

In [ ]:
df_ped.withColumnRenamed("Valor Tabela", "valor_tabela")\
.withColumnRenamed("% de desconto", "perc_desconto")\
.withColumnRenamed("Peso Líquido", "peso_liquido")\
.withColumnRenamed("Peso Bruto", "peso_bruto").show(1, False)

+----------+-----------+---------+-------+----------+-------------------------------+--------+---+----------------------------------------------+---------+-------------------+-------------+-------------+--------+-------------------------------------------------+------+-----+----------+-----------+------------+-------------+------------+----------+
|NotaFiscal|Rota       |Documento|Pedido |CodCliente|Cliente                        |Cidade  |UF |Filial                                        |Quadrante|GrupoItem          |DataDocumento|Representante|CodItem |Item                                             |UNNF  |Qtd  |FatorVenda|FatorCompra|valor_tabela|perc_desconto|peso_liquido|peso_bruto|
+----------+-----------+---------+-------+----------+-------------------------------+--------+---+----------------------------------------------+---------+-------------------+-------------+-------------+--------+-------------------------------------------------+------+-----+----------+-----------+--

In [ ]:
df_ped = df_ped.withColumnRenamed("Valor Tabela", "valor_tabela")\
.withColumnRenamed("% de desconto", "perc_desconto")\
.withColumnRenamed("Peso Líquido", "peso_liquido")\
.withColumnRenamed("Peso Bruto", "peso_bruto")

In [ ]:
df_ped.printSchema()

root
 |-- NotaFiscal: integer (nullable = true)
 |-- Rota: string (nullable = true)
 |-- Documento: integer (nullable = true)
 |-- Pedido: integer (nullable = true)
 |-- CodCliente: string (nullable = true)
 |-- Cliente: string (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- Filial: string (nullable = true)
 |-- Quadrante: string (nullable = true)
 |-- GrupoItem: string (nullable = true)
 |-- DataDocumento: string (nullable = true)
 |-- Representante: string (nullable = true)
 |-- CodItem: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- UNNF: string (nullable = true)
 |-- Qtd: double (nullable = true)
 |-- FatorVenda: string (nullable = true)
 |-- FatorCompra: string (nullable = true)
 |-- valor_tabela: double (nullable = true)
 |-- perc_desconto: double (nullable = true)
 |-- peso_liquido: double (nullable = true)
 |-- peso_bruto: double (nullable = true)



In [ ]:
df_ped.withColumn("DataDocumento", f.to_date(df_ped["DataDocumento"], "MM/dd/yyyy")).show(1, False)

+----------+-----------+---------+-------+----------+-------------------------------+--------+---+----------------------------------------------+---------+-------------------+-------------+-------------+--------+-------------------------------------------------+------+-----+----------+-----------+------------+-------------+------------+----------+
|NotaFiscal|Rota       |Documento|Pedido |CodCliente|Cliente                        |Cidade  |UF |Filial                                        |Quadrante|GrupoItem          |DataDocumento|Representante|CodItem |Item                                             |UNNF  |Qtd  |FatorVenda|FatorCompra|valor_tabela|perc_desconto|peso_liquido|peso_bruto|
+----------+-----------+---------+-------+----------+-------------------------------+--------+---+----------------------------------------------+---------+-------------------+-------------+-------------+--------+-------------------------------------------------+------+-----+----------+-----------+--

In [ ]:
df_ped = df_ped.withColumn("DataDocumento", f.to_date(df_ped["DataDocumento"], "MM/dd/yyyy"))

# Até aqui só trabalhei na limpeza dos dados, garantindo que as informações
# estivessem no padrão de dados para trabalhar. Exemplo: as datas que estavam
# como string transformamos para toDate e os valores que estavam errados corrigimos
# daqui pra frente, iremos montar os formatos de tabelas que precisamos de acordo
# com o que se é necessário para gerar as comissões dos internos.

In [ ]:
# Transformando os DataFrames em TempViews, que são a forma de criar uma tabela
# possível de ser lida com códigos SQL. A partir daqui vamos começar a fazer
# consultas e criar novas colunas calculadas.

df_com.createOrReplaceTempView("v_comiss")
df_dev.createOrReplaceTempView("v_dev")
df_fat.createOrReplaceTempView("v_fat")
df_ped.createOrReplaceTempView("v_ped")

In [ ]:
df_com.show(1, False)

+----------+------------------------+------------+----------+---------+-------------+-------------+-------+--------------------------------+----+----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+
|CodCliente|cliente                 |pedido_venda|NotaFiscal|documento|DataDocumento|Representante|CodItem|item                            |UNNF|Qtd |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|
+----------+------------------------+------------+----------+---------+-------------+-------------+-------+--------------------------------+----+----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+
|CL008563  |FABRICIA ROWEDER ANTONIO|NULL        |2172      |2042678  |2026-02-10   |MAR E RIO    |6      |FILE MERLUZA A

In [ ]:
#df_com.withColumn("chave",
#                  f.concat_ws(",", f.col("CodCliente"), f.col("DataDocumento"), f.col("CodItem"), f.col("Qtd"))
#                  ).show(1, False)

In [ ]:
# Visualizando as tabelas criadas no Spark com códigos SQL, fazendo um INNER JOIN
# diretamente na tabela pedidos para trazer as informações de qual pedido é a devida
# comissão.
spark.sql(

          """
          SELECT c.*, p.`perc_desconto`
          FROM v_comiss c
          INNER JOIN v_ped p on p.Pedido = c.pedido_venda
          """
).show(2, False)

+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+-------------+
|CodCliente|cliente                           |pedido_venda|NotaFiscal|documento|DataDocumento|Representante   |CodItem|item                  |UNNF|Qtd|ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|perc_desconto|
+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+-------------+
|CL000713  |BRAZILIAN GOURMET RESTAURANTE LTDA|1797905     |16202     |20

In [ ]:
#spark.sql(
#
 #         """
  #        SELECT c.CodCliente, c.cliente, c.ValorContabil,
   #       CASE
    #        WHEN p.perc_desconto between -10000 and 4.00 THEN c.ValorContabil * 0.03
     #       WHEN p.perc_desconto between 4.01 and 9.00 THEN c.ValorContabil * 0.025
      #      ELSE c.ValorContabil * 0.015
       #   END AS valor_comiss,
        #  p.Pedido, p.perc_desconto
         # FROM v_comiss c
          #INNER JOIN v_ped p ON p.Pedido = c.pedido_venda
          #GROUP BY c.CodCliente, c.cliente, p.Pedido, c.ValorContabil, p.perc_desconto
 #         """
#). show(1, False)

In [ ]:
# Aqui nós criamos um modelo de coluna com os valores de comissão baseado nas regras
# que a empresa tem, porém como se trata dos internos que a regra é diferente, esse cálculo
# vai ficar para quando formos gerar a comissão dos externos.

spark.sql(

          """
          SELECT c.CodCliente, c.cliente, c.pedido_venda, c.NotaFiscal,
            c.documento, c.DataDocumento, c.Representante, c.CodItem, c.item,
            c.UNNF, c.Qtd, c.ValorContabil, c.comissao_no_pv, c.tabela_promocional,
            c.U_CodItem, c.RS_tabela_comissao, c.percent_comissao, c.comissao_padrao_3perc,
            c.total_documento, c.tipo_item,
            CASE
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN -10000.00 AND 4.00 THEN c.ValorContabil * 0.06
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN 6.01 AND 9.00 THEN c.ValorContabil * 0.04
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN 4.01 AND 6.00 THEN c.ValorContabil * 0.05
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN 9.01 AND 10000.00 THEN c.ValorContabil * 0.03
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN -10000.00 AND 4.00 THEN c.ValorContabil * 0.03
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 4.01 AND 6.00 THEN c.ValorContabil * 0.025
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 6.01 AND 9.00 THEN c.ValorContabil * 0.02
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 9.01 AND 12.00 THEN c.ValorContabil * 0.015
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 12.01 AND 10000.00 THEN c.ValorContabil * 0.01
              WHEN c.tipo_item LIKE 'Salm%' AND p.perc_desconto BETWEEN -10000.00 AND 4.00 THEN c.ValorContabil * 0.01
              WHEN c.tipo_item LIKE 'Salm%' AND p.perc_desconto BETWEEN 4.01 AND 6.00 THEN c.ValorContabil * 0.0075
              WHEN c.tipo_item LIKE 'Salm%' AND p.perc_desconto BETWEEN 6.01 AND 10000.00 THEN c.ValorContabil * 0.005
              ELSE 0.0
            END AS valor_pag_comiss,
            p.perc_desconto
            FROM v_comiss c
            INNER JOIN v_ped p ON p.Pedido = c.pedido_venda
          """
).show(2, False)

+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+------------------+-------------+
|CodCliente|cliente                           |pedido_venda|NotaFiscal|documento|DataDocumento|Representante   |CodItem|item                  |UNNF|Qtd|ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|valor_pag_comiss  |perc_desconto|
+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+------------------+-------------+
|CL000713  |BRAZ

In [ ]:
# Aqui atribuimos a consulta feita anteriormente a um DataFrame para não perder
# as informações.

v_comiss_ordenado = spark.sql(

          """
          SELECT c.CodCliente, c.cliente, c.pedido_venda, c.NotaFiscal,
            c.documento, c.DataDocumento, c.Representante, c.CodItem, c.item,
            c.UNNF, c.Qtd, c.ValorContabil, c.comissao_no_pv, c.tabela_promocional,
            c.U_CodItem, c.RS_tabela_comissao, c.percent_comissao, c.comissao_padrao_3perc,
            c.total_documento, c.tipo_item,
            CASE
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN -10000.00 AND 4.00 THEN c.ValorContabil * 0.06
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN 6.01 AND 9.00 THEN c.ValorContabil * 0.04
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN 4.01 AND 6.00 THEN c.ValorContabil * 0.05
              WHEN c.tipo_item LIKE 'Bebid%' AND p.perc_desconto BETWEEN 9.01 AND 10000.00 THEN c.ValorContabil * 0.03
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN -10000.00 AND 4.00 THEN c.ValorContabil * 0.03
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 4.01 AND 6.00 THEN c.ValorContabil * 0.025
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 6.01 AND 9.00 THEN c.ValorContabil * 0.02
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 9.01 AND 12.00 THEN c.ValorContabil * 0.015
              WHEN c.tipo_item LIKE 'Mix%' AND p.perc_desconto BETWEEN 12.01 AND 10000.00 THEN c.ValorContabil * 0.01
              WHEN c.tipo_item LIKE 'Salm%' AND p.perc_desconto BETWEEN -10000.00 AND 4.00 THEN c.ValorContabil * 0.01
              WHEN c.tipo_item LIKE 'Salm%' AND p.perc_desconto BETWEEN 4.01 AND 6.00 THEN c.ValorContabil * 0.0075
              WHEN c.tipo_item LIKE 'Salm%' AND p.perc_desconto BETWEEN 6.01 AND 10000.00 THEN c.ValorContabil * 0.005
              ELSE 0.0
            END AS valor_pag_comiss,
            p.perc_desconto
            FROM v_comiss c
            INNER JOIN v_ped p ON p.Pedido = c.pedido_venda
          """
)

In [ ]:
# Transformei esse DF em uma TempView para fazer uma nova consulta
v_comiss_ordenado.createOrReplaceTempView("v_comiss_ord")

In [ ]:
# Aqui finalmente aplico as regras de comissão dos vendedores internos, juntamente
# com os cálculos a serem realizados para cada venda deles.

spark.sql(
    """
    SELECT co.*,
            CASE
              WHEN co.tipo_item like 'Mix%' THEN '0.40%'
              WHEN co.tipo_item like 'Salm%' THEN '0.15%'
              WHEN co.tipo_item like 'Bebid%' THEN '0.1%'
              ELSE '0%'
            END AS perc_comiss,
            CASE
              WHEN co.Representante like '%INTERNO' AND co.tipo_item like 'Mix%' THEN co.ValorContabil * 0.004
              WHEN co.Representante like '%INTERNO' AND co.tipo_item like 'Salm%' THEN co.ValorContabil * 0.0015
              WHEN co.Representante like '%INTERNO' AND co.tipo_item like 'Bebid%' THEN co.ValorContabil * 0.001
              ELSE co.ValorContabil * 0
            END AS comiss_int
      FROM v_comiss_ord co
    """

).show(1, False)

+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+----------------+-------------+-----------+----------+
|CodCliente|cliente                           |pedido_venda|NotaFiscal|documento|DataDocumento|Representante   |CodItem|item                  |UNNF|Qtd|ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|valor_pag_comiss|perc_desconto|perc_comiss|comiss_int|
+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+--------

In [ ]:
# Atribuo então esses dados novamente ao DataFrame
v_comiss_ordenado2 = spark.sql(
    """
    SELECT co.*,
            CASE
              WHEN co.tipo_item like 'Mix%' THEN '0.40%'
              WHEN co.tipo_item like 'Salm%' THEN '0.15%'
              WHEN co.tipo_item like 'Bebid%' THEN '0.1%'
              ELSE '0%'
            END AS perc_comiss,
            CASE
              WHEN co.Representante like '%INTERNO' AND co.tipo_item like 'Mix%' THEN co.ValorContabil * 0.004
              WHEN co.Representante like '%INTERNO' AND co.tipo_item like 'Salm%' THEN co.ValorContabil * 0.0015
              WHEN co.Representante like '%INTERNO' AND co.tipo_item like 'Bebid%' THEN co.ValorContabil * 0.001
              ELSE co.ValorContabil * 0
            END AS comiss_int
      FROM v_comiss_ord co
    """

)

In [ ]:
# Aqui como não ia precisar dessa coluna (que só vai ser utilizado na comissão externa)
# então removi a coluna inteira usando o método .drop()
v_comiss_ordenado2 = v_comiss_ordenado2.drop("valor_pag_comiss")

In [ ]:
# Visualiando se a alteração ficou correta sem a coluna removida
v_comiss_ordenado2.show(1, False)

+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+-------------+-----------+----------+
|CodCliente|cliente                           |pedido_venda|NotaFiscal|documento|DataDocumento|Representante   |CodItem|item                  |UNNF|Qtd|ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|perc_desconto|perc_comiss|comiss_int|
+----------+----------------------------------+------------+----------+---------+-------------+----------------+-------+----------------------+----+---+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+-------------+-----------+----------+
|CL0

In [ ]:
# Aqui fiz uma pequena alteração no nome da coluna para ficar intuitivo
v_comiss_ordenado2 = v_comiss_ordenado2.withColumnRenamed("perc_comiss", "perc_comiss_int")

In [ ]:
# Novamente peguei os dados do meu DF e transformei em um TempView
v_comiss_ordenado2.createOrReplaceTempView("v_comiss_geral")

In [ ]:
# Fiz o passo anterior para que nessa consulta eu pudesse usar o Where trazendo somente o que são vendedores internos
spark.sql(
    """
    SELECT *
    FROM v_comiss_geral cg
    WHERE cg.Representante like '%INTERNO'
    """
).show(10, False)

+----------+-------------------------------------------+------------+----------+---------+-------------+-----------------+--------+------------------------------------------------+----+----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+-------------+---------------+------------------+
|CodCliente|cliente                                    |pedido_venda|NotaFiscal|documento|DataDocumento|Representante    |CodItem |item                                            |UNNF|Qtd |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|perc_desconto|perc_comiss_int|comiss_int        |
+----------+-------------------------------------------+------------+----------+---------+-------------+-----------------+--------+------------------------------------------------+----+----+-------------+--------------+-----------------

In [ ]:
# Aqui então coloquei novamente esses dados no DataFrame
v_comiss_ordenado2 = spark.sql(
    """
    SELECT *
    FROM v_comiss_geral cg
    WHERE cg.Representante like '%INTERNO'
    """
)

In [ ]:
v_comiss_ordenado2.createOrReplaceTempView("v_comiss_int_final")

In [ ]:
spark.sql(
    """
    SELECT *
    FROM v_comiss_int_final
    """
).show(4, False)

+----------+-------------------------------------+------------+----------+---------+-------------+-----------------+--------+------------------------------------------------+----+----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+-------------+---------------+----------+
|CodCliente|cliente                              |pedido_venda|NotaFiscal|documento|DataDocumento|Representante    |CodItem |item                                            |UNNF|Qtd |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|perc_desconto|perc_comiss_int|comiss_int|
+----------+-------------------------------------+------------+----------+---------+-------------+-----------------+--------+------------------------------------------------+----+----+-------------+--------------+------------------+---------+------------------+---

In [ ]:
comiss_interno_vd_final = spark.sql(
    """
    SELECT *
    FROM v_comiss_int_final
    """
)

In [ ]:
distribuidor.createOrReplaceTempView("v_dist")

In [ ]:
comiss_interno_vd_final = spark.sql(
    """
      SELECT cf.*,
      CASE
        WHEN cf.Representante LIKE '%MAR E RIO%' THEN "N"
        ELSE cf.Representante
      END AS Rep,
      CASE
        WHEN TRIM(dt.` SAP`) = cf.CodCliente THEN "Dist"
        ELSE "Não Dist"
      END AS Distribuidor
      FROM v_comiss_int_final cf
      LEFT JOIN v_dist dt ON TRIM(dt.` SAP`) = cf.CodCliente
    """
)

In [ ]:
# Aqui é uma redundância, durante o processo acabei refazendo passsos, mas é basicamente
# a mesma consulta que estou visualizando.

# Aqui terminamos a parte de vendas das comissões internas, então iremos partir
# para as devoluções após aqui, que será uma nova consulta em outra tabela.

comiss_interno_vd_final.show(2, False)

+----------+-----------------------+------------+----------+---------+-------------+-----------------+--------+------------------------------------------------+----+----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+-------------+---------------+----------+-----------------+------------+
|CodCliente|cliente                |pedido_venda|NotaFiscal|documento|DataDocumento|Representante    |CodItem |item                                            |UNNF|Qtd |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|perc_desconto|perc_comiss_int|comiss_int|Rep              |Distribuidor|
+----------+-----------------------+------------+----------+---------+-------------+-----------------+--------+------------------------------------------------+----+----+-------------+--------------+------------------+---------+--

In [ ]:
spark.sql(
    """
    SELECT dt.` SAP`
    FROM v_dist dt
    WHERE dt.` SAP` = ' CL003057'
    """
).show()

+---------+
|      SAP|
+---------+
| CL003057|
+---------+



In [ ]:
# Aqui eu já fui direto para as devoluções, como estavam finalzadas as vendas
# Esse passo é importante, por que no valor final da comissão deles precisa
# descontar o valor de devoluções que tiveram.

# Aqui atribui a tabela colunas de cálculo e referencia que tinhamos nos modelos
# manuais do excel. Assim quando exportarmos os dados, já estará tudo
spark.sql(
    """
    SELECT *
    FROM
      (SELECT d.*,
      SUBSTRING(d.ItemCode, 1, 2) AS Id,
      p.Pedido,
      CASE
        WHEN d.Dscription LIKE 'SALMAO EVISC%' THEN "Salmão"
        ELSE "Mix"
      END AS Tipo,
      CASE
        WHEN d.Representante LIKE '%MAR E RIO%' THEN "N"
        ELSE d.Representante
      END AS RepValida,
      CASE
        WHEN Id NOT LIKE '%RV%'
          AND Id NOT LIKE '%MP%'
          AND Id NOT LIKE '%PA%'
        THEN -(d.Valor*0.03)
        ELSE -(d.comissao_no_pv + d.tabela_promocional + d.RS_tabela_comissao + d.comissao_padrao_3perc)
      END AS Comissao,
      (Comissao / d.Valor)*-1 AS Perc_Comissao
      FROM v_dev d
      INNER JOIN v_ped p ON p.NotaFiscal = d.NF) AS v_dev
      WHERE v_dev.Representante LIKE '%INTERNO'
    """
).show(1, False)

+--------+----------------------+------+-------+----------+--------+-------+---------------+--------+---------------------------------------------------------+-------+-------+------+--------------+------------------+------------------+--------------------+---------------------+------------------+--------------+-------+---+-------+----+--------------+--------+--------------------+
|cardcode|cardname              |NumDev|NFDev  |data_dev  |nf_saida|NF     |status_dev     |ItemCode|Dscription                                               |UomCode|qtd_dev|Valor |comissao_no_pv|tabela_promocional|RS_tabela_comissao|perc_tabela_comissao|comissao_padrao_3perc|valor_comissao_dev|Representante |LineNum|Id |Pedido |Tipo|RepValida     |Comissao|Perc_Comissao       |
+--------+----------------------+------+-------+----------+--------+-------+---------------+--------+---------------------------------------------------------+-------+-------+------+--------------+------------------+------------------

In [ ]:
# Aqui já estou atribuindo a um DataFrame para ter as informações da consulta.

comiss_dev_int = spark.sql(
    """
    SELECT *
    FROM
      (SELECT d.*,
      SUBSTRING(d.ItemCode, 1, 2) AS Id,
      p.Pedido,
      CASE
        WHEN d.Dscription LIKE 'SALMAO EVISC%' THEN "Salmão"
        ELSE "Mix"
      END AS Tipo,
      CASE
        WHEN d.Representante LIKE '%MAR E RIO%' THEN "N"
        ELSE d.Representante
      END AS RepValida,
      CASE
        WHEN Id NOT LIKE '%RV%'
          AND Id NOT LIKE '%MP%'
          AND Id NOT LIKE '%PA%'
        THEN -(d.Valor*0.03)
        ELSE -(d.comissao_no_pv + d.tabela_promocional + d.RS_tabela_comissao + d.comissao_padrao_3perc)
      END AS Comissao,
      (Comissao / d.Valor)*-1 AS Perc_Comissao
      FROM v_dev d
      INNER JOIN v_ped p ON p.NotaFiscal = d.NF) AS v_dev
      WHERE v_dev.Representante LIKE '%INTERNO'
    """
)

In [ ]:
# Visualizando se o DataFrame está com as inforamções necessárias.
comiss_dev_int.show(3, False)

+--------+-------------------------------------+------+-------+----------+--------+-------+---------------+--------+-----------------------------------------------------------------+-------+-------+------+--------------+------------------+------------------+--------------------+---------------------+------------------+---------------+-------+---+-------+----+---------------+--------+--------------------+
|cardcode|cardname                             |NumDev|NFDev  |data_dev  |nf_saida|NF     |status_dev     |ItemCode|Dscription                                                       |UomCode|qtd_dev|Valor |comissao_no_pv|tabela_promocional|RS_tabela_comissao|perc_tabela_comissao|comissao_padrao_3perc|valor_comissao_dev|Representante  |LineNum|Id |Pedido |Tipo|RepValida      |Comissao|Perc_Comissao       |
+--------+-------------------------------------+------+-------+----------+--------+-------+---------------+--------+-----------------------------------------------------------------+--

In [ ]:
comiss_dev_int.createOrReplaceTempView("comiss_dev_int")


In [ ]:
spark.sql(
    """
    SELECT TRIM(` SAP`) AS SAP
    FROM v_dist
    """
).show(1, False)

+--------+
|SAP     |
+--------+
|CL007109|
+--------+
only showing top 1 row


In [ ]:
comiss_dev_int = spark.sql(
    """
    SELECT cd.*,
    CASE
      WHEN TRIM(dt.` SAP`) = TRIM(cd.cardcode) THEN 'Dist'
      ELSE 'Não Dist'
    END AS Dist
    FROM comiss_dev_int cd
    LEFT JOIN v_dist dt ON TRIM(dt.` SAP`) = cd.cardcode
    --WHERE cd.cardname LIKE '%FOOD%'
    """
)

In [ ]:
comiss_dev_int.createOrReplaceTempView("comiss_dev_int")


In [ ]:
spark.sql(
    """
    SELECT c.cardcode, c.cardname, c.NumDev, c.NFDev, c.data_dev, c.nf_saida,
           c.NF, c.status_dev, c.ItemCode, c.Dscription, c.UomCode, c.qtd_dev,
           c.Valor, c.comissao_no_pv, c.tabela_promocional, c.RS_tabela_comissao,
           c.perc_tabela_comissao, c.comissao_padrao_3perc, c.valor_comissao_dev,
           c.Representante, c.LineNum, c.Id, c.Pedido, c.Tipo, c.RepValida,
           c.Comissao, c.Perc_Comissao, c.Dist
    FROM comiss_dev_int c
    GROUP BY c.cardcode, c.cardname, c.NumDev, c.NFDev, c.data_dev, c.nf_saida,
             c.NF, c.status_dev, c.ItemCode, c.Dscription, c.UomCode, c.qtd_dev,
             c.Valor, c.comissao_no_pv, c.tabela_promocional, c.RS_tabela_comissao,
             c.perc_tabela_comissao, c.comissao_padrao_3perc, c.valor_comissao_dev,
             c.Representante, c.LineNum, c.Id, c.Pedido, c.Tipo, c.RepValida,
             c.Comissao, c.Perc_Comissao, c.Dist
    """
).show(40, False)

+--------+------------------------------------------------+------+-------+----------+--------+-------+---------------+--------+-------------------------------------------------------------------+-------+-------+-------+--------------+------------------+------------------+--------------------+---------------------+------------------+---------------+-------+---+-------+------+---------------+--------+---------------------+--------+
|cardcode|cardname                                        |NumDev|NFDev  |data_dev  |nf_saida|NF     |status_dev     |ItemCode|Dscription                                                         |UomCode|qtd_dev|Valor  |comissao_no_pv|tabela_promocional|RS_tabela_comissao|perc_tabela_comissao|comissao_padrao_3perc|valor_comissao_dev|Representante  |LineNum|Id |Pedido |Tipo  |RepValida      |Comissao|Perc_Comissao        |Dist    |
+--------+------------------------------------------------+------+-------+----------+--------+-------+---------------+--------+-----

In [ ]:
comiss_dev_int = spark.sql(
    """
    SELECT c.cardcode, c.cardname, c.NumDev, c.NFDev, c.data_dev, c.nf_saida,
           c.NF, c.status_dev, c.ItemCode, c.Dscription, c.UomCode, c.qtd_dev,
           c.Valor, c.comissao_no_pv, c.tabela_promocional, c.RS_tabela_comissao,
           c.perc_tabela_comissao, c.comissao_padrao_3perc, c.valor_comissao_dev,
           c.Representante, c.LineNum, c.Id, c.Pedido, c.Tipo, c.RepValida,
           c.Comissao, c.Perc_Comissao, c.Dist
    FROM comiss_dev_int c
    GROUP BY c.cardcode, c.cardname, c.NumDev, c.NFDev, c.data_dev, c.nf_saida,
             c.NF, c.status_dev, c.ItemCode, c.Dscription, c.UomCode, c.qtd_dev,
             c.Valor, c.comissao_no_pv, c.tabela_promocional, c.RS_tabela_comissao,
             c.perc_tabela_comissao, c.comissao_padrao_3perc, c.valor_comissao_dev,
             c.Representante, c.LineNum, c.Id, c.Pedido, c.Tipo, c.RepValida,
             c.Comissao, c.Perc_Comissao, c.Dist
    """
)

In [ ]:
comiss_interno_vd_final.show(1, False)
comiss_dev_int.show(1, False)

+----------+--------------------------------------+------------+----------+---------+-------------+---------------+--------+------------------------------------------+----+-----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+----------+---------------+---+-------+----------+------------+
|CodCliente|cliente                               |pedido_venda|NotaFiscal|documento|DataDocumento|Representante  |CodItem |item                                      |UNNF|Qtd  |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|comiss_int|Rep            |Id |Pedido |Repeticoes|Distribuidor|
+----------+--------------------------------------+------------+----------+---------+-------------+---------------+--------+------------------------------------------+----+-----+-------------+--------------+------------------+------

In [ ]:
comiss_interno_vd_final.show(1, False)


+----------+--------------------------------------+------------+----------+---------+-------------+---------------+--------+------------------------------------------+----+-----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+----------+---------------+---+-------+----------+------------+
|CodCliente|cliente                               |pedido_venda|NotaFiscal|documento|DataDocumento|Representante  |CodItem |item                                      |UNNF|Qtd  |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|comiss_int|Rep            |Id |Pedido |Repeticoes|Distribuidor|
+----------+--------------------------------------+------------+----------+---------+-------------+---------------+--------+------------------------------------------+----+-----+-------------+--------------+------------------+------

In [ ]:
comiss_interno_vd_final.createOrReplaceTempView("comiss_interno_vd_final")

In [ ]:
spark.sql(
    """
    SELECT cf.CodCliente, cf.cliente, cf.pedido_venda, cf.NotaFiscal, cf.documento,
    cf.DataDocumento, cf.Representante, cf.CodItem, cf.item, cf.UNNF, cf.Qtd, cf.ValorContabil,
    cf.comissao_no_pv, cf.tabela_promocional, cf.U_CodItem, cf.RS_tabela_comissao,
    cf.percent_comissao, cf.comissao_padrao_3perc, cf.total_documento, cf.tipo_item,
    cf.comiss_int, cf.Rep,
    SUBSTRING(cf.CodItem, 1, 2) AS Id,
    p.Pedido,
    COUNT(*) OVER (PARTITION BY cf.NotaFiscal, cf.documento, cf.Representante, cf.CodItem, cf.Qtd, cf.ValorContabil) AS Repeticoes,
    cf.Distribuidor
    FROM comiss_interno_vd_final cf
    LEFT JOIN v_ped p ON p.NotaFiscal = cf.NotaFiscal
    GROUP BY cf.CodCliente, cf.cliente, cf.pedido_venda, cf.NotaFiscal, cf.documento,
    cf.DataDocumento, cf.Representante, cf.CodItem, cf.item, cf.UNNF, cf.Qtd, cf.ValorContabil,
    cf.comissao_no_pv, cf.tabela_promocional, cf.U_CodItem, cf.RS_tabela_comissao,
    cf.percent_comissao, cf.comissao_padrao_3perc, cf.total_documento, cf.tipo_item,
    cf.comiss_int, cf.Rep, p.Pedido, cf.Distribuidor
    """
).show(20, False)

+----------+--------------------------------------+------------+----------+---------+-------------+-----------------+--------+--------------------------------------------------------------+----+-----+-------------+--------------+------------------+---------+------------------+----------------+---------------------+---------------+---------+------------------+-----------------+---+-------+----------+------------+
|CodCliente|cliente                               |pedido_venda|NotaFiscal|documento|DataDocumento|Representante    |CodItem |item                                                          |UNNF|Qtd  |ValorContabil|comissao_no_pv|tabela_promocional|U_CodItem|RS_tabela_comissao|percent_comissao|comissao_padrao_3perc|total_documento|tipo_item|comiss_int        |Rep              |Id |Pedido |Repeticoes|Distribuidor|
+----------+--------------------------------------+------------+----------+---------+-------------+-----------------+--------+------------------------------------------

In [ ]:
comiss_interno_vd_final = spark.sql(
    """
    SELECT cf.CodCliente, cf.cliente, cf.pedido_venda, cf.NotaFiscal, cf.documento,
    cf.DataDocumento, cf.Representante, cf.CodItem, cf.item, cf.UNNF, cf.Qtd, cf.ValorContabil,
    cf.comissao_no_pv, cf.tabela_promocional, cf.U_CodItem, cf.RS_tabela_comissao,
    cf.percent_comissao, cf.comissao_padrao_3perc, cf.total_documento, cf.tipo_item,
    cf.comiss_int, cf.Rep,
    SUBSTRING(cf.CodItem, 1, 2) AS Id,
    p.Pedido,
    COUNT(*) OVER (PARTITION BY cf.NotaFiscal, cf.documento, cf.Representante, cf.CodItem, cf.Qtd, cf.ValorContabil) AS Repeticoes,
    cf.Distribuidor
    FROM comiss_interno_vd_final cf
    LEFT JOIN v_ped p ON p.NotaFiscal = cf.NotaFiscal
    GROUP BY cf.CodCliente, cf.cliente, cf.pedido_venda, cf.NotaFiscal, cf.documento,
    cf.DataDocumento, cf.Representante, cf.CodItem, cf.item, cf.UNNF, cf.Qtd, cf.ValorContabil,
    cf.comissao_no_pv, cf.tabela_promocional, cf.U_CodItem, cf.RS_tabela_comissao,
    cf.percent_comissao, cf.comissao_padrao_3perc, cf.total_documento, cf.tipo_item,
    cf.comiss_int, cf.Rep, p.Pedido, cf.Distribuidor
    """
)

In [ ]:
pdf1 = comiss_interno_vd_final.toPandas()

In [ ]:
pdf1.to_excel("/content/drive/MyDrive/internos_export/vd_interno.xlsx", index=False)

In [ ]:
pdf2 = comiss_dev_int.toPandas()

In [ ]:
pdf2.to_excel("/content/drive/MyDrive/internos_export/dev_interno.xlsx", index=False)

In [ ]:
with pd.ExcelWriter("/content/drive/MyDrive/internos_export/comissao_internos.xlsx", engine="xlsxwriter") as writer:
  pdf1.to_excel(writer, sheet_name = "Geral_Comissão Item_MAR26", index=False)
  pdf2.to_excel(writer, sheet_name = "Geral-dev-comissao_MAR26_", index=False)

  workbook = writer.book
  worksheet1 = writer.sheets["Geral_Comissão Item_MAR26"]
  worksheet2 = writer.sheets["Geral-dev-comissao_MAR26_"]

  header_format = workbook.add_format({
        "bold": True,
        "text_wrap": True,
        "valign": "top",
        "fg_color": "#D7E4BC",
        "border": 1
  })

  for col_num, value in enumerate(pdf1.columns.values):
        worksheet1.write(0, col_num, value, header_format)
        worksheet1.set_column(col_num, col_num, 20)

  for col_num, value in enumerate(pdf2.columns.values):
        worksheet2.write(0, col_num, value, header_format)
        worksheet2.set_column(col_num, col_num, 20)